# 04 — Curva de Juros (ETTJ), Inflação Implícita e Decomposição de Risco

Este notebook demonstra um fluxo completo de análise macro-quantitativa sobre dados reais do **Tesouro Direto** (extraídos da fixture oficial `tests/fixtures/tesouro_oficial_sample.csv`, datas-base `11/09/2026` e `18/09/2026`):

1. **Construção da ETTJ Prefixada e IPCA+ (Juro Real)** em anos úteis reais (`calendario.dias_uteis / 252`), comparando a interpolação **flat-forward** (padrão B3/ANBIMA) e **Nelson-Siegel**;
2. **Estrutura a termo da inflação implícita (Breakeven IPCA)** via identidade exata de Fisher (`pre_para_ipca_mais`);
3. **Decomposição de retorno semanal (`11/09/2026` → `18/09/2026`)** entre **carrego (`pu_curva`)** e **marcação a mercado (`pu_mercado`)** via `marktomarket.curva(..., ettj=...)`, confrontando com o `PU Compra Manhã` oficial do Tesouro;
4. **Aproximação de Taylor de 1ª ordem (`Duration Modificada`) vs. 2ª ordem (`Duration + Convexidade`)** contra a re-precificação exata sob choques paralelos de taxa.

In [1]:
from datetime import date

from rfquant.comparison import taxa_equivalente
from rfquant.data import calendario
from rfquant.pricing import marktomarket, precificar, risco
from rfquant.pricing.curva_juros import ETTJ

# 1. Vértices reais do Tesouro Direto em 18/09/2026 (tests/fixtures/tesouro_oficial_sample.csv)
data_base = date(2026, 9, 18)

vertices_pre_18set = [
    (date(2027, 1, 1), 0.1354),
    (date(2028, 1, 1), 0.1360),
    (date(2029, 1, 1), 0.1385),
    (date(2031, 1, 1), 0.1410),
    (date(2032, 1, 1), 0.1419),
]
vertices_ipca_18set = [
    (date(2029, 5, 15), 0.0737),
    (date(2032, 8, 15), 0.0758),
    (date(2035, 5, 15), 0.0757),
    (date(2040, 8, 15), 0.0727),
    (date(2045, 5, 15), 0.0726),
    (date(2050, 8, 15), 0.0719),
]

pontos_pre = [
    (calendario.dias_uteis(data_base, venc) / 252.0, taxa)
    for venc, taxa in vertices_pre_18set
]
pontos_ipca = [
    (calendario.dias_uteis(data_base, venc) / 252.0, taxa)
    for venc, taxa in vertices_ipca_18set
]

ettj_pre_ff = ETTJ(pontos_pre, metodo='flat_forward')
ettj_pre_ns = ETTJ(pontos_pre, metodo='nelson_siegel')
ettj_ipca_ff = ETTJ(pontos_ipca, metodo='flat_forward')

print('=== ETTJ Prefixada em 18/09/2026: Flat-Forward (B3/ANBIMA) vs. Nelson-Siegel ===')
print(f"{'Prazo (anos DU)':<16} {'Flat-Forward':>14} {'Nelson-Siegel':>15} {'Diferença (bps)':>17} {'Fator Desc. (FF)':>18}")
for t in [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]:
    y_ff = ettj_pre_ff.taxa(t)
    y_ns = ettj_pre_ns.taxa(t)
    df_ff = ettj_pre_ff.fator_desconto(t)
    diff_bps = (y_ff - y_ns) * 10_000
    print(f'{t:<16.1f} {y_ff:>14.4%} {y_ns:>15.4%} {diff_bps:>+17.1f} {df_ff:>18.6f}')

=== ETTJ Prefixada em 18/09/2026: Flat-Forward (B3/ANBIMA) vs. Nelson-Siegel ===
Prazo (anos DU)    Flat-Forward   Nelson-Siegel   Diferença (bps)   Fator Desc. (FF)
0.5                    13.5741%        13.4844%              +9.0           0.938340
1.0                    13.5954%        13.5342%              +6.1           0.880317
2.0                    13.8082%        13.7878%              +2.0           0.772063
3.0                    13.9820%        13.9763%              +0.6           0.675291
4.0                    14.0825%        14.0924%              -1.0           0.590369
5.0                    14.1712%        14.1661%              +0.5           0.515487


In [2]:
# 2. Estrutura a termo da Inflação Implícita (Breakeven IPCA via Fisher exata)
# (1 + y_pre) = (1 + y_real) * (1 + pi_implicita)
print('=== Inflação Implícita (Breakeven IPCA) por Vértice — 18/09/2026 ===')
print(f"{'Prazo (anos DU)':<16} {'Nominal (Pré)':>15} {'Real (IPCA+)':>14} {'IPCA Implícito':>16} {'Prova Fisher':>14}")
for t in [2.5, 3.0, 3.5, 4.0, 4.5, 5.0]:
    y_nom = ettj_pre_ff.taxa(t)
    y_real = ettj_ipca_ff.taxa(t)
    breakeven = taxa_equivalente.pre_para_ipca_mais(y_nom, y_real)
    prova_nom = taxa_equivalente.ipca_mais_para_pre(y_real, breakeven)
    print(f'{t:<16.1f} {y_nom:>15.4%} {y_real:>14.4%} {breakeven:>16.4%} {prova_nom:>14.4%}')

=== Inflação Implícita (Breakeven IPCA) por Vértice — 18/09/2026 ===
Prazo (anos DU)    Nominal (Pré)   Real (IPCA+)   IPCA Implícito   Prova Fisher
2.5                     13.9017%        7.3700%          6.0833%       13.9017%
3.0                     13.9820%        7.4177%          6.1110%       13.9820%
3.5                     14.0394%        7.4651%          6.1176%       14.0394%
4.0                     14.0825%        7.5007%          6.1226%       14.0825%
4.5                     14.1266%        7.5284%          6.1363%       14.1266%
5.0                     14.1712%        7.5506%          6.1558%       14.1712%


In [3]:
# 3. Decomposição de Retorno Semanal (11/09/2026 -> 18/09/2026) via marktomarket.curva
# Compramos os 5 Tesouro Prefixado (LTN) em 11/09/2026 pelas taxas oficiais de compra
# e marcamos a mercado em 18/09/2026 contra a curva ETTJ de 18/09/2026.
carteira_ltn = [
    {
        'nome': 'Prefixado 2027',
        'data_compra': date(2026, 9, 11),
        'data_vencimento': date(2027, 1, 1),
        'taxa_compra': 0.1357,
        'pu_oficial_18set': 965.34,
    },
    {
        'nome': 'Prefixado 2028',
        'data_compra': date(2026, 9, 11),
        'data_vencimento': date(2028, 1, 1),
        'taxa_compra': 0.1355,
        'pu_oficial_18set': 850.07,
    },
    {
        'nome': 'Prefixado 2029',
        'data_compra': date(2026, 9, 11),
        'data_vencimento': date(2029, 1, 1),
        'taxa_compra': 0.1377,
        'pu_oficial_18set': 746.11,
    },
    {
        'nome': 'Prefixado 2031',
        'data_compra': date(2026, 9, 11),
        'data_vencimento': date(2031, 1, 1),
        'taxa_compra': 0.1400,
        'pu_oficial_18set': 571.16,
    },
    {
        'nome': 'Prefixado 2032',
        'data_compra': date(2026, 9, 11),
        'data_vencimento': date(2032, 1, 1),
        'taxa_compra': 0.1408,
        'pu_oficial_18set': 498.51,
    },
]

resultados_mtm = marktomarket.curva(
    data_base,
    carteira_ltn,
    ettj=ettj_pre_ff,
)

print('=== Decomposição Carrego vs. Marcação a Mercado (11/09/2026 -> 18/09/2026, 5 DU) ===')
cab = f"{'Título':<16} {'PU 11/09':>10} {'PU Curva':>10} {'PU Merc.':>10} {'PU Oficial':>11} {'Carrego':>9} {'Efeito MtM':>11} {'Total (%)':>10}"
print(cab)
for item, res in zip(carteira_ltn, resultados_mtm, strict=True):
    carrego_rs = res.pu_curva - res.pu_compra
    mtm_rs = res.variacao_absoluta
    ret_total = (res.pu_mercado / res.pu_compra) - 1.0
    print(
        f"{item['nome']:<16} {res.pu_compra:>10.2f} {res.pu_curva:>10.2f} "
        f"{res.pu_mercado:>10.2f} {item['pu_oficial_18set']:>11.2f} "
        f"{carrego_rs:>+9.2f} {mtm_rs:>+11.2f} {ret_total:>+10.2%}"
    )

=== Decomposição Carrego vs. Marcação a Mercado (11/09/2026 -> 18/09/2026, 5 DU) ===
Título             PU 11/09   PU Curva   PU Merc.  PU Oficial   Carrego  Efeito MtM  Total (%)
Prefixado 2027       962.84     965.27     965.34      965.34     +2.43       +0.07     +0.26%
Prefixado 2028       848.41     850.55     850.08      850.07     +2.14       -0.48     +0.20%
Prefixado 2029       745.39     747.30     746.11      746.11     +1.91       -1.19     +0.10%
Prefixado 2031       571.81     573.30     571.17      571.16     +1.49       -2.13     -0.11%
Prefixado 2032       499.74     501.05     498.52      498.51     +1.31       -2.53     -0.24%


In [4]:
# 4. Sensibilidade de 1ª ordem (Duration Modificada) vs. 2ª ordem (Duration + Convexidade)
# Título real: Tesouro Prefixado com Juros Semestrais 2037 em 18/09/2026 (cupom 10% a.a., taxa 14,25%)
venc_ntnf = date(2037, 1, 1)
y0 = 0.1425
av_base = precificar(data_base, venc_ntnf, y0, taxa_cupom_anual=0.10)
print(
    f'NTN-F 2037 em 18/09/2026 -> PU: R$ {av_base.pu:,.2f} (oficial: R$ 808,19) | '
    f'D_mod: {av_base.duration_modificada:.4f} | Convexidade: {av_base.convexidade:.4f} | '
    f'DV01: R$ {av_base.dv01:.4f}/bp'
)
print()
print(f"{'Choque (bps)':<14} {'1ª Ordem (D_mod)':>18} {'2ª Ordem (D_mod+Conv)':>23} {'Exata (Re-precif.)':>20} {'Erro 2ª Ordem (bps)':>21}")
for choque_bps in [-200, -100, -50, 50, 100, 200]:
    dy = choque_bps / 10_000.0
    var_1a = -av_base.duration_modificada * dy
    var_2a = risco.variacao_pu(av_base.duration_modificada, av_base.convexidade, dy)
    pu_chocado = precificar(data_base, venc_ntnf, y0 + dy, taxa_cupom_anual=0.10).pu
    var_exata = (pu_chocado / av_base.pu) - 1.0
    erro_bps = (var_2a - var_exata) * 10_000
    print(f'{choque_bps:<+14d} {var_1a:>+18.4%} {var_2a:>+23.4%} {var_exata:>+20.4%} {erro_bps:>+21.2f}')

NTN-F 2037 em 18/09/2026 -> PU: R$ 808.11 (oficial: R$ 808,19) | D_mod: 5.2712 | Convexidade: 42.9764 | DV01: R$ 0.4258/bp

Choque (bps)     1ª Ordem (D_mod)   2ª Ordem (D_mod+Conv)   Exata (Re-precif.)   Erro 2ª Ordem (bps)
-200                    +10.5425%               +11.4020%            +11.4610%                 -5.89
-100                     +5.2712%                +5.4861%             +5.4933%                 -0.72
-50                      +2.6356%                +2.6893%             +2.6902%                 -0.09
+50                      -2.6356%                -2.5819%             -2.5828%                 +0.09
+100                     -5.2712%                -5.0564%             -5.0631%                 +0.68
+200                    -10.5425%                -9.6830%             -9.7358%                 +5.28
